# 实验 009：以正式模型为锚点的近期专家走步融合

本 Notebook 使用已经通过线上检验的 `legacy_328 + LightGBM LambdaRank` 作为锚点，在 Train 内三折走步验证中选择近期 1702 期专家的轮数与融合权重。实验结果只写入自己的结果目录，**绝不自动覆盖** `04_results/final_submission/prediction.npy`。


## tl;dr

- 默认 `DSCR_EXP009_MODE=full`：执行三折训练、官方 Valid 一次性安全检查、Train+Valid 近期专家重训和完整 Test 预测。
- 每次完整运行都会生成 `prediction.npy`、`expert_prediction.npy`、`valid_prediction.npy`、`metrics.json`、`metadata.json`、CSV 明细和中文报告。
- 如果没有候选通过 Train 内稳定性门槛，权重自动回退到 `0`，仍会输出一个与正式锚点一致的标准预测文件。
- `DSCR_EXP009_MODE=preflight` 只用于快速验证数据契约、选择逻辑和结果写出；产物写入 `04_results/exp_009_anchor_recent_blend_cv/preflight/`。


## Context & Methods

### Key Assumptions

1. `processed_data_v1` 的前 328 列与 `exp_003` 的稳定特征视图兼容，且 `READY`、manifest SHA-256 和 legacy compatibility 均通过。
2. 所有训练、权重选择和轮数选择只使用 Train 内时间折；官方 Valid 只做一次安全检查，不反向修改权重。
3. Test 锚点直接读取当前正式提交；新实验仅训练近期专家，避免重训过程改变已验证锚点。
4. 每个时间截面先独立转百分位秩再融合，非评价位置严格填充 `0.5`。


## Setup

### 1. 参数、路径与运行模式


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data.z").exists() and (candidate / "02_experiments").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录：请从项目根目录或实验目录启动 Notebook。")


@dataclass(frozen=True)
class WalkForwardFold:
    name: str
    train_start: int
    train_stop: int
    valid_start: int
    valid_stop: int


PROJECT_ROOT = find_project_root()
EXPERIMENT_ID = "exp_009_anchor_recent_blend_cv"
DATASET_DIR = PROJECT_ROOT / "03_cache" / "processed_data_v1"
OUTPUT_DIR = PROJECT_ROOT / "04_results" / EXPERIMENT_ID
MANIFEST_PATH = DATASET_DIR / "manifest.json"
READY_PATH = DATASET_DIR / "READY"
ANCHOR_PATH = PROJECT_ROOT / "04_results" / "final_submission" / "prediction.npy"
ANCHOR_METADATA_PATH = PROJECT_ROOT / "04_results" / "final_submission" / "metadata.json"

RUN_MODE = os.environ.get("DSCR_EXP009_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "preflight"}:
    raise ValueError("DSCR_EXP009_MODE 只允许 full 或 preflight。")
RUN_DIR = OUTPUT_DIR if RUN_MODE == "full" else OUTPUT_DIR / "preflight"
RUNTIME_CACHE_DIR = RUN_DIR / "runtime_cache"

TRAIN_START, TRAIN_STOP = 486, 2918
VALID_START, VALID_STOP = 2918, 3161
TEST_START, TEST_STOP = 3161, 3603
TEST_TIME_POINTS, STOCK_COUNT = 442, 5282
FEATURE_STOP = 328
TRAIN_STOCK_CAP = 1200
RECENT_LOOKBACK = 1702
BASE_ROUNDS = 8
RECENT_ROUNDS = (8, 16)
RECENT_WEIGHTS = (0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.35)
SEED = 42
NUM_THREADS = max(1, (os.cpu_count() or 8) - 2)

FOLDS = (
    WalkForwardFold("fold_1", 486, 2189, 2189, 2432),
    WalkForwardFold("fold_2", 486, 2432, 2432, 2675),
    WalkForwardFold("fold_3", 486, 2675, 2675, 2918),
)

MIN_POSITIVE_FOLDS = 2
MIN_MEAN_IMPROVEMENT = 0.0005
MAX_WORST_FOLD_DROP = 0.0015
MAX_LATE_MEAN_DROP = 0.0003
OFFICIAL_MIN_IMPROVEMENT = 0.0003
OFFICIAL_MAX_LATE_DROP = 0.0002
OFFICIAL_MAX_WORST_QUARTER_DROP = 0.0015
ANCHOR_EXPECTED_VALID_IC = 0.09294016824452567
ANCHOR_REPRODUCTION_TOLERANCE = 0.0003
MIN_TEST_ANCHOR_CORRELATION = 0.97

random.seed(SEED)
np.random.seed(SEED)
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(pd.Series({
    "project_root": str(PROJECT_ROOT),
    "run_mode": RUN_MODE,
    "run_dir": str(RUN_DIR),
    "features": FEATURE_STOP,
    "stock_cap": TRAIN_STOCK_CAP,
    "recent_lookback": RECENT_LOOKBACK,
    "recent_rounds": RECENT_ROUNDS,
    "weights": RECENT_WEIGHTS,
    "threads": NUM_THREADS,
}))


## Data

### 2. 验证重型缓存并加载固定视图


In [ ]:
def file_sha256(path: Path, block_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    partial.write_text(text, encoding="utf-8")
    os.replace(partial, path)


def atomic_write_json(path: Path, payload: dict) -> None:
    atomic_write_text(path, json.dumps(payload, ensure_ascii=False, indent=2))


def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    with partial.open("wb") as handle:
        np.save(handle, array)
    os.replace(partial, path)


def atomic_save_npz(path: Path, **arrays: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    with partial.open("wb") as handle:
        np.savez(handle, **arrays)
    os.replace(partial, path)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    frame.to_csv(partial, index=False, encoding="utf-8-sig")
    os.replace(partial, path)


if not READY_PATH.exists() or not MANIFEST_PATH.exists():
    raise RuntimeError("processed_data_v1 缺少 READY 或 manifest.json，禁止训练。")

ready = json.loads(READY_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_sha256 = file_sha256(MANIFEST_PATH)

assert manifest["status"] == "ready"
assert ready["manifest_sha256"] == manifest_sha256
assert manifest["dimensions"] == {"time": 3603, "stock": 5282, "raw_numeric": 99, "raw_category": 9}
assert manifest["splits"]["train"]["start"] == TRAIN_START
assert manifest["splits"]["train"]["stop"] == TRAIN_STOP
assert manifest["splits"]["valid"]["start"] == VALID_START
assert manifest["splits"]["valid"]["stop"] == VALID_STOP
assert manifest["splits"]["test"]["start"] == TEST_START
assert manifest["splits"]["test"]["stop"] == TEST_STOP
assert manifest["features"]["legacy_numeric_prefix"] == FEATURE_STOP
assert manifest["features"]["tree_count"] == 419
assert manifest["legacy_compatibility"]["status"] == "passed"
assert manifest["validation"]["test_mask"]["count"] == 2_042_538
assert manifest["validation"]["test_mask"]["matches_official"] is True


def load_common(split: str) -> dict[str, np.ndarray]:
    directory = DATASET_DIR / "common"
    result = {
        "time": np.load(directory / f"{split}_time.npy", mmap_mode="r"),
        "stock": np.load(directory / f"{split}_stock.npy", mmap_mode="r"),
        "groups": np.load(directory / f"{split}_group_sizes.npy", mmap_mode="r"),
    }
    if split != "test":
        result["y"] = np.load(directory / f"{split}_y.npy", mmap_mode="r")
        result["relevance"] = np.load(directory / f"{split}_relevance.npy", mmap_mode="r")
    return result


def load_tree(split: str) -> np.ndarray:
    matrix = np.load(DATASET_DIR / "tree" / f"{split}_X.npy", mmap_mode="r")
    expected_rows = int(manifest["expected_rows"][split])
    assert matrix.shape == (expected_rows, 419)
    return matrix


common = {split: load_common(split) for split in ("train", "valid", "test")}
tree = {split: load_tree(split) for split in ("train", "valid", "test")}

data_rows = []
for split in ("train", "valid", "test"):
    values = common[split]
    assert int(values["groups"].sum()) == values["time"].size == values["stock"].size
    assert np.all(np.diff(values["time"]) >= 0)
    assert np.isfinite(tree[split][:32, :FEATURE_STOP]).all()
    data_rows.append({
        "split": split,
        "rows": int(values["time"].size),
        "time_start": int(values["time"][0]),
        "time_stop": int(values["time"][-1]) + 1,
        "time_points": int(values["groups"].size),
        "group_sum": int(values["groups"].sum()),
    })

assert ANCHOR_PATH.exists() and ANCHOR_METADATA_PATH.exists()
anchor_metadata = json.loads(ANCHOR_METADATA_PATH.read_text(encoding="utf-8"))
assert anchor_metadata["prediction_sha256"] == file_sha256(ANCHOR_PATH)

DATA_CONTRACT = pd.DataFrame(data_rows)
display(DATA_CONTRACT)
print("缓存和正式锚点数据契约：通过。")


### 3. 指标、时间切片和截面秩工具


In [ ]:
def rank_ic(prediction: np.ndarray, target: np.ndarray) -> float:
    prediction = np.asarray(prediction)
    target = np.asarray(target)
    usable = np.isfinite(prediction) & np.isfinite(target)
    if int(usable.sum()) < 2:
        return np.nan
    x = rankdata(prediction[usable])
    y = rankdata(target[usable])
    if x.std() == 0 or y.std() == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def group_rank_ic_series(prediction: np.ndarray, target: np.ndarray, groups: np.ndarray) -> np.ndarray:
    prediction = np.asarray(prediction)
    target = np.asarray(target)
    values = []
    offset = 0
    for size in groups:
        size = int(size)
        values.append(rank_ic(prediction[offset:offset + size], target[offset:offset + size]))
        offset += size
    assert offset == prediction.size == target.size
    return np.asarray(values, dtype=np.float64)


def group_rank_transform(prediction: np.ndarray, groups: np.ndarray) -> np.ndarray:
    prediction = np.asarray(prediction)
    ranked = np.empty(prediction.size, dtype=np.float32)
    offset = 0
    for size in groups:
        size = int(size)
        ranked[offset:offset + size] = rankdata(
            prediction[offset:offset + size], method="average"
        ).astype(np.float32) / float(size)
        offset += size
    assert offset == prediction.size
    return ranked


def score_prediction(prediction: np.ndarray, target: np.ndarray, groups: np.ndarray) -> dict[str, float]:
    per_time = group_rank_ic_series(prediction, target, groups)
    quarters = np.array_split(per_time, 4)
    return {
        "mean_rank_ic": float(np.nanmean(per_time)),
        "std_rank_ic": float(np.nanstd(per_time)),
        "late_half_rank_ic": float(np.nanmean(per_time[len(per_time) // 2:])),
        "worst_quarter_rank_ic": float(min(np.nanmean(part) for part in quarters)),
        "negative_time_share": float(np.mean(per_time < 0)),
    }


def row_slice_for_times(split: str, start_time: int, stop_time: int) -> tuple[slice, np.ndarray]:
    times = common[split]["time"]
    start = int(np.searchsorted(times, start_time, side="left"))
    stop = int(np.searchsorted(times, stop_time, side="left"))
    split_start = int(times[0])
    group_start = start_time - split_start
    group_stop = stop_time - split_start
    groups = np.asarray(common[split]["groups"][group_start:group_stop], dtype=np.int32)
    assert int(groups.sum()) == stop - start
    return slice(start, stop), groups


def capped_indices_for_split(
    split: str,
    start_time: int,
    stop_time: int,
    cap: int,
) -> tuple[np.ndarray, np.ndarray]:
    row_slice, full_groups = row_slice_for_times(split, start_time, stop_time)
    capped_groups = np.minimum(full_groups, int(cap)).astype(np.int32)
    indices = np.empty(int(capped_groups.sum()), dtype=np.int64)
    source_offset = int(row_slice.start)
    target_offset = 0
    for full_size, capped_size in zip(full_groups, capped_groups):
        full_size = int(full_size)
        capped_size = int(capped_size)
        positions = np.linspace(0, full_size - 1, capped_size, dtype=np.int64)
        indices[target_offset:target_offset + capped_size] = source_offset + positions
        source_offset += full_size
        target_offset += capped_size
    assert source_offset == int(row_slice.stop)
    assert target_offset == indices.size == int(capped_groups.sum())
    return indices, capped_groups


def mean_cross_sectional_rank_correlation(
    left_grid: np.ndarray,
    right_grid: np.ndarray,
    groups: np.ndarray,
    stocks: np.ndarray,
) -> float:
    values = []
    offset = 0
    for local_time, size in enumerate(groups):
        size = int(size)
        current_stocks = np.asarray(stocks[offset:offset + size], dtype=np.int32)
        values.append(rank_ic(left_grid[local_time, current_stocks], right_grid[local_time, current_stocks]))
        offset += size
    assert offset == stocks.size
    return float(np.nanmean(values))


assert abs(rank_ic(np.arange(10), np.arange(10)) - 1.0) < 1e-12
toy_groups = np.array([4, 3], dtype=np.int32)
toy_prediction = np.array([4, 1, 3, 2, 1, 3, 2], dtype=np.float32)
toy_ranked = group_rank_transform(toy_prediction, toy_groups)
assert toy_ranked.shape == toy_prediction.shape
assert np.all((toy_ranked > 0) & (toy_ranked <= 1))
print("指标、时间切片和截面秩工具自检通过。")


### 4. LightGBM 训练、分块预测和断点复用


In [ ]:
LGB_PARAMS = {
    "objective": "lambdarank",
    "metric": "None",
    "learning_rate": 0.0228695,
    "num_leaves": 79,
    "min_data_in_leaf": 147,
    "feature_fraction": 0.80936,
    "bagging_fraction": 0.647764,
    "bagging_freq": 1,
    "lambda_l1": 2.35724,
    "lambda_l2": 0.238705,
    "max_bin": 127,
    "label_gain": list(range(64)),
    "lambdarank_truncation_level": 1024,
    "verbosity": -1,
    "seed": SEED,
    "feature_fraction_seed": SEED,
    "bagging_seed": SEED,
    "num_threads": NUM_THREADS,
}

fingerprint_payload = {
    "experiment_id": EXPERIMENT_ID,
    "manifest_sha256": manifest_sha256,
    "feature_stop": FEATURE_STOP,
    "stock_cap": TRAIN_STOCK_CAP,
    "recent_lookback": RECENT_LOOKBACK,
    "base_rounds": BASE_ROUNDS,
    "recent_rounds": RECENT_ROUNDS,
    "weights": RECENT_WEIGHTS,
    "folds": [asdict(fold) for fold in FOLDS],
    "lgb_params": LGB_PARAMS,
}
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]


def build_training_arrays(
    segments: list[tuple[str, int, int]],
    cap: int = TRAIN_STOCK_CAP,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    prepared = []
    total_rows = 0
    for split, start_time, stop_time in segments:
        indices, groups = capped_indices_for_split(split, start_time, stop_time, cap)
        prepared.append((split, indices, groups))
        total_rows += indices.size

    X_train = np.empty((total_rows, FEATURE_STOP), dtype=np.float32)
    y_train = np.empty(total_rows, dtype=np.int8)
    all_groups = []
    offset = 0
    for split, indices, groups in prepared:
        stop = offset + indices.size
        X_train[offset:stop] = tree[split][indices, :FEATURE_STOP]
        y_train[offset:stop] = common[split]["relevance"][indices]
        all_groups.append(groups)
        offset = stop
    assert offset == total_rows
    final_groups = np.concatenate(all_groups).astype(np.int32)
    assert int(final_groups.sum()) == total_rows
    return X_train, y_train, final_groups


def train_ranker(
    segments: list[tuple[str, int, int]],
    boost_rounds: int,
):
    import lightgbm as lgb

    X_train, y_train, train_groups = build_training_arrays(segments)
    dataset = lgb.Dataset(
        X_train,
        label=y_train,
        group=train_groups,
        free_raw_data=True,
    )
    started_at = time.time()
    model = lgb.train(
        LGB_PARAMS,
        dataset,
        num_boost_round=int(boost_rounds),
        callbacks=[lgb.log_evaluation(0)],
    )
    elapsed = time.time() - started_at
    train_rows = int(X_train.shape[0])
    del dataset, X_train, y_train, train_groups
    gc.collect()
    return model, {"training_seconds": elapsed, "train_rows": train_rows}


def predict_interval(
    model,
    split: str,
    start_time: int,
    stop_time: int,
    num_iteration: int,
    chunk_size: int = 200_000,
) -> tuple[np.ndarray, np.ndarray]:
    rows, groups = row_slice_for_times(split, start_time, stop_time)
    prediction = np.empty(int(rows.stop) - int(rows.start), dtype=np.float32)
    for begin in range(int(rows.start), int(rows.stop), chunk_size):
        end = min(begin + chunk_size, int(rows.stop))
        prediction[begin - int(rows.start):end - int(rows.start)] = model.predict(
            tree[split][begin:end, :FEATURE_STOP],
            num_iteration=int(num_iteration),
        ).astype(np.float32)
    return prediction, groups


def interval_target(split: str, start_time: int, stop_time: int) -> np.ndarray:
    rows, _ = row_slice_for_times(split, start_time, stop_time)
    return np.asarray(common[split]["y"][rows], dtype=np.float32)


def save_model_text_atomic(model, path: Path) -> None:
    atomic_write_text(path, model.model_to_string())


print("模型参数指纹：", RUN_FINGERPRINT)


In [ ]:
if RUN_MODE == "preflight":
    smoke_model, smoke_train_info = train_ranker(
        [("train", 486, 490)],
        boost_rounds=2,
    )
    smoke_prediction, smoke_groups = predict_interval(
        smoke_model,
        "train",
        490,
        492,
        num_iteration=2,
    )
    smoke_target = interval_target("train", 490, 492)
    assert smoke_prediction.size == smoke_target.size == int(smoke_groups.sum())
    assert np.isfinite(smoke_prediction).all()
    smoke_metrics = score_prediction(smoke_prediction, smoke_target, smoke_groups)
    del smoke_model
    gc.collect()
    print({
        "real_cache_lightgbm_smoke": "passed",
        "train_rows": smoke_train_info["train_rows"],
        "prediction_rows": smoke_prediction.size,
        "mean_rank_ic": smoke_metrics["mean_rank_ic"],
    })
else:
    print("完整模式：跳过重复的轻量冒烟测试，直接进入走步训练。")


## Results

### 5. Train 内三折走步验证

每个折都训练一个完整历史锚点和一个最多 16 轮的近期专家；近期专家在第 8/16 轮分别预测，因此无需为两个轮数重复构造训练矩阵。每折预测写入带配置指纹的断点缓存，重复运行会直接复用。


In [ ]:
def synthetic_preflight_fold_results() -> pd.DataFrame:
    rows = []
    baseline_values = [0.0880, 0.0930, 0.0970]
    for fold, baseline in zip(FOLDS, baseline_values):
        rows.append({
            "fold": fold.name,
            "recent_rounds": 0,
            "recent_weight": 0.0,
            "mean_rank_ic": baseline,
            "std_rank_ic": 0.10,
            "late_half_rank_ic": baseline - 0.001,
            "worst_quarter_rank_ic": baseline - 0.030,
            "negative_time_share": 0.20,
            "mean_delta": 0.0,
            "late_delta": 0.0,
        })
        for rounds in RECENT_ROUNDS:
            round_bonus = 0.00010 if rounds == 16 else 0.0
            for weight in RECENT_WEIGHTS[1:]:
                improvement = 0.00120 - abs(weight - 0.15) * 0.006 + round_bonus
                improvement += {"fold_1": -0.00010, "fold_2": 0.00005, "fold_3": 0.00010}[fold.name]
                rows.append({
                    "fold": fold.name,
                    "recent_rounds": rounds,
                    "recent_weight": weight,
                    "mean_rank_ic": baseline + improvement,
                    "std_rank_ic": 0.10,
                    "late_half_rank_ic": baseline - 0.001 + improvement * 0.8,
                    "worst_quarter_rank_ic": baseline - 0.030 + improvement * 0.5,
                    "negative_time_share": 0.20,
                    "mean_delta": improvement,
                    "late_delta": improvement * 0.8,
                })
    return pd.DataFrame(rows)


if RUN_MODE == "preflight":
    fold_results = synthetic_preflight_fold_results()
    print("Preflight：使用合成折结果验证选择与写出逻辑。")
else:
    fold_rows = []
    for fold in FOLDS:
        fold_cache_path = RUNTIME_CACHE_DIR / f"{fold.name}_{RUN_FINGERPRINT}.npz"
        valid_target = interval_target("train", fold.valid_start, fold.valid_stop)
        _, valid_groups = row_slice_for_times("train", fold.valid_start, fold.valid_stop)

        if fold_cache_path.exists():
            cached = np.load(fold_cache_path)
            base_prediction = np.asarray(cached["base_prediction"], dtype=np.float32)
            recent_prediction_8 = np.asarray(cached["recent_prediction_8"], dtype=np.float32)
            recent_prediction_16 = np.asarray(cached["recent_prediction_16"], dtype=np.float32)
            expected = valid_target.size
            assert base_prediction.size == recent_prediction_8.size == recent_prediction_16.size == expected
            print(f"复用 {fold.name} 预测缓存。")
        else:
            print(f"{fold.name}：训练完整历史锚点。", flush=True)
            base_model, base_train_info = train_ranker(
                [("train", fold.train_start, fold.train_stop)],
                BASE_ROUNDS,
            )
            base_prediction, _ = predict_interval(
                base_model, "train", fold.valid_start, fold.valid_stop, BASE_ROUNDS
            )
            del base_model
            gc.collect()

            recent_start = max(fold.train_start, fold.train_stop - RECENT_LOOKBACK)
            print(f"{fold.name}：训练近期专家 [{recent_start}, {fold.train_stop})。", flush=True)
            recent_model, recent_train_info = train_ranker(
                [("train", recent_start, fold.train_stop)],
                max(RECENT_ROUNDS),
            )
            recent_prediction_8, _ = predict_interval(
                recent_model, "train", fold.valid_start, fold.valid_stop, 8
            )
            recent_prediction_16, _ = predict_interval(
                recent_model, "train", fold.valid_start, fold.valid_stop, 16
            )
            del recent_model
            gc.collect()
            atomic_save_npz(
                fold_cache_path,
                base_prediction=base_prediction,
                recent_prediction_8=recent_prediction_8,
                recent_prediction_16=recent_prediction_16,
            )

        base_rank = group_rank_transform(base_prediction, valid_groups)
        base_metrics = score_prediction(base_rank, valid_target, valid_groups)
        fold_rows.append({
            "fold": fold.name,
            "recent_rounds": 0,
            "recent_weight": 0.0,
            **base_metrics,
            "mean_delta": 0.0,
            "late_delta": 0.0,
        })

        for rounds, recent_prediction in (
            (8, recent_prediction_8),
            (16, recent_prediction_16),
        ):
            recent_rank = group_rank_transform(recent_prediction, valid_groups)
            for weight in RECENT_WEIGHTS[1:]:
                blended = (1.0 - weight) * base_rank + weight * recent_rank
                blend_metrics = score_prediction(blended, valid_target, valid_groups)
                fold_rows.append({
                    "fold": fold.name,
                    "recent_rounds": rounds,
                    "recent_weight": weight,
                    **blend_metrics,
                    "mean_delta": blend_metrics["mean_rank_ic"] - base_metrics["mean_rank_ic"],
                    "late_delta": blend_metrics["late_half_rank_ic"] - base_metrics["late_half_rank_ic"],
                })

        fold_results = pd.DataFrame(fold_rows)
        atomic_write_csv(RUN_DIR / "fold_results.csv", fold_results)
        display(fold_results[fold_results["fold"] == fold.name].sort_values("mean_rank_ic", ascending=False).head(8))

fold_results = fold_results.sort_values(["fold", "recent_rounds", "recent_weight"]).reset_index(drop=True)
atomic_write_csv(RUN_DIR / "fold_results.csv", fold_results)
print(f"走步结果行数：{len(fold_results)}")


### 6. 受约束选择近期轮数和权重


In [ ]:
summary_rows = []
for (rounds, weight), frame in fold_results.groupby(["recent_rounds", "recent_weight"]):
    summary_rows.append({
        "recent_rounds": int(rounds),
        "recent_weight": float(weight),
        "mean_rank_ic": float(frame["mean_rank_ic"].mean()),
        "fold_std": float(frame["mean_rank_ic"].std(ddof=0)),
        "late_half_rank_ic": float(frame["late_half_rank_ic"].mean()),
        "worst_quarter_rank_ic": float(frame["worst_quarter_rank_ic"].min()),
        "mean_improvement": float(frame["mean_delta"].mean()),
        "worst_fold_delta": float(frame["mean_delta"].min()),
        "late_mean_delta": float(frame["late_delta"].mean()),
        "positive_fold_count": int((frame["mean_delta"] > 0).sum()),
    })

weight_search = pd.DataFrame(summary_rows)
weight_search["qualified"] = (
    (weight_search["recent_weight"] > 0)
    & (weight_search["recent_weight"] <= 0.25)
    & (weight_search["positive_fold_count"] >= MIN_POSITIVE_FOLDS)
    & (weight_search["mean_improvement"] >= MIN_MEAN_IMPROVEMENT)
    & (weight_search["worst_fold_delta"] >= -MAX_WORST_FOLD_DROP)
    & (weight_search["late_mean_delta"] >= -MAX_LATE_MEAN_DROP)
)

qualified = weight_search[weight_search["qualified"]].copy()
if qualified.empty:
    selected_recent_rounds = 8
    selected_recent_weight = 0.0
    selection_reason = "没有候选通过 Train 内稳定性门槛，回退到正式锚点。"
else:
    best_improvement = float(qualified["mean_improvement"].max())
    near_best = qualified[
        qualified["mean_improvement"] >= best_improvement * 0.95
    ].sort_values(
        ["recent_weight", "fold_std", "recent_rounds"],
        ascending=[True, True, True],
    )
    selected = near_best.iloc[0]
    selected_recent_rounds = int(selected["recent_rounds"])
    selected_recent_weight = float(selected["recent_weight"])
    selection_reason = "选择达到最佳平均增益 95% 范围内的最低近期权重。"

if selected_recent_weight == 0.0:
    weight_search["selected"] = (
        (weight_search["recent_rounds"] == 0)
        & np.isclose(weight_search["recent_weight"], 0.0)
    )
else:
    weight_search["selected"] = (
        (weight_search["recent_rounds"] == selected_recent_rounds)
        & np.isclose(weight_search["recent_weight"], selected_recent_weight)
    )
atomic_write_csv(RUN_DIR / "weight_search.csv", weight_search)

display(weight_search.sort_values(["qualified", "mean_improvement"], ascending=[False, False]).head(15))
print(pd.Series({
    "selected_recent_rounds": selected_recent_rounds,
    "selected_recent_weight": selected_recent_weight,
    "selection_reason": selection_reason,
}))


### 7. 官方 Valid 一次性安全检查

选定轮数和权重后，使用完整 Train 重训锚点与近期专家，在官方 Valid 上只检查一次。该检查只决定 `promoted` 状态，不会搜索或修改权重。


In [ ]:
def prediction_vector_to_grid(
    prediction: np.ndarray,
    split: str,
    time_start: int,
    time_points: int,
) -> np.ndarray:
    grid = np.full((time_points, STOCK_COUNT), 0.5, dtype=np.float32)
    times = np.asarray(common[split]["time"], dtype=np.int32)
    stocks = np.asarray(common[split]["stock"], dtype=np.int32)
    grid[times - time_start, stocks] = prediction
    return grid


if RUN_MODE == "preflight":
    official_base_metrics = {
        "mean_rank_ic": ANCHOR_EXPECTED_VALID_IC,
        "std_rank_ic": 0.10,
        "late_half_rank_ic": 0.0918,
        "worst_quarter_rank_ic": 0.0600,
        "negative_time_share": 0.20,
    }
    official_blend_metrics = {
        **official_base_metrics,
        "mean_rank_ic": ANCHOR_EXPECTED_VALID_IC + 0.0006,
        "late_half_rank_ic": 0.0922,
        "worst_quarter_rank_ic": 0.0602,
    }
    official_expert_metrics = {
        **official_base_metrics,
        "mean_rank_ic": 0.0940,
    }
    official_anchor_reproduced = True
    valid_grid = np.full((243, STOCK_COUNT), 0.5, dtype=np.float32)
    official_valid_cache_used = False
else:
    official_cache_path = RUNTIME_CACHE_DIR / (
        f"official_valid_{RUN_FINGERPRINT}_r{selected_recent_rounds}.npz"
    )
    valid_target = np.asarray(common["valid"]["y"], dtype=np.float32)
    valid_groups = np.asarray(common["valid"]["groups"], dtype=np.int32)

    if official_cache_path.exists():
        cached = np.load(official_cache_path)
        official_base_prediction = np.asarray(cached["base_prediction"], dtype=np.float32)
        official_expert_prediction = np.asarray(cached["expert_prediction"], dtype=np.float32)
        assert official_base_prediction.size == official_expert_prediction.size == valid_target.size
        official_valid_cache_used = True
        print("复用官方 Valid 预测缓存。")
    else:
        print("训练官方 Valid 锚点模型。", flush=True)
        official_base_model, _ = train_ranker(
            [("train", TRAIN_START, TRAIN_STOP)],
            BASE_ROUNDS,
        )
        official_base_prediction, _ = predict_interval(
            official_base_model, "valid", VALID_START, VALID_STOP, BASE_ROUNDS
        )
        del official_base_model
        gc.collect()

        official_recent_start = TRAIN_STOP - RECENT_LOOKBACK
        print(f"训练官方 Valid 近期专家 [{official_recent_start}, {TRAIN_STOP})。", flush=True)
        official_expert_model, _ = train_ranker(
            [("train", official_recent_start, TRAIN_STOP)],
            selected_recent_rounds,
        )
        official_expert_prediction, _ = predict_interval(
            official_expert_model,
            "valid",
            VALID_START,
            VALID_STOP,
            selected_recent_rounds,
        )
        del official_expert_model
        gc.collect()
        atomic_save_npz(
            official_cache_path,
            base_prediction=official_base_prediction,
            expert_prediction=official_expert_prediction,
        )
        official_valid_cache_used = False

    official_base_rank = group_rank_transform(official_base_prediction, valid_groups)
    official_expert_rank = group_rank_transform(official_expert_prediction, valid_groups)
    official_blend_rank = (
        (1.0 - selected_recent_weight) * official_base_rank
        + selected_recent_weight * official_expert_rank
    )
    official_base_metrics = score_prediction(official_base_rank, valid_target, valid_groups)
    official_expert_metrics = score_prediction(official_expert_rank, valid_target, valid_groups)
    official_blend_metrics = score_prediction(official_blend_rank, valid_target, valid_groups)
    official_anchor_reproduced = (
        abs(official_base_metrics["mean_rank_ic"] - ANCHOR_EXPECTED_VALID_IC)
        <= ANCHOR_REPRODUCTION_TOLERANCE
    )
    valid_grid = prediction_vector_to_grid(
        official_blend_rank, "valid", VALID_START, VALID_STOP - VALID_START
    )

official_mean_delta = official_blend_metrics["mean_rank_ic"] - official_base_metrics["mean_rank_ic"]
official_late_delta = official_blend_metrics["late_half_rank_ic"] - official_base_metrics["late_half_rank_ic"]
official_worst_delta = official_blend_metrics["worst_quarter_rank_ic"] - official_base_metrics["worst_quarter_rank_ic"]

promoted = bool(
    RUN_MODE == "full"
    and selected_recent_weight > 0
    and official_anchor_reproduced
    and official_mean_delta >= OFFICIAL_MIN_IMPROVEMENT
    and official_late_delta >= -OFFICIAL_MAX_LATE_DROP
    and official_worst_delta >= -OFFICIAL_MAX_WORST_QUARTER_DROP
)

official_valid_results = pd.DataFrame([
    {"model": "anchor", **official_base_metrics},
    {"model": "recent_expert", **official_expert_metrics},
    {"model": "selected_blend", **official_blend_metrics},
])
atomic_write_csv(RUN_DIR / "official_valid_results.csv", official_valid_results)
atomic_save_npy(RUN_DIR / "valid_prediction.npy", valid_grid)
display(official_valid_results)
print(pd.Series({
    "anchor_reproduced": official_anchor_reproduced,
    "official_mean_delta": official_mean_delta,
    "official_late_delta": official_late_delta,
    "official_worst_quarter_delta": official_worst_delta,
    "promoted": promoted,
}))


### 8. Train+Valid 近期专家重训与完整 Test 预测

Test 锚点直接读取正式提交。近期专家只训练一次；预测先逐时间截面排名，再按选定权重融合并再次排名。即使 `promoted=False`，本实验仍保存自己的 `prediction.npy`，但不会改动正式提交。


In [ ]:
anchor_grid = np.load(ANCHOR_PATH)
assert anchor_grid.shape == (TEST_TIME_POINTS, STOCK_COUNT)
assert anchor_grid.dtype == np.float32
assert np.isfinite(anchor_grid).all()

if RUN_MODE == "preflight":
    expert_grid = anchor_grid.copy()
    prediction_grid = anchor_grid.copy()
    test_anchor_correlation = 1.0
    final_model_reused = False
    print("Preflight：复制正式锚点用于验证完整结果写出。")
else:
    import lightgbm as lgb

    final_recent_start = VALID_STOP - RECENT_LOOKBACK
    final_model_path = RUN_DIR / "model_recent.txt"
    final_model_metadata_path = RUN_DIR / "model_recent.metadata.json"
    final_model_fingerprint = hashlib.sha256(
        json.dumps({
            "run_fingerprint": RUN_FINGERPRINT,
            "train_start": final_recent_start,
            "train_stop": VALID_STOP,
            "rounds": selected_recent_rounds,
        }, sort_keys=True).encode("utf-8")
    ).hexdigest()

    final_model_reused = False
    if final_model_path.exists() and final_model_metadata_path.exists():
        saved_model_metadata = json.loads(final_model_metadata_path.read_text(encoding="utf-8"))
        if saved_model_metadata.get("fingerprint") == final_model_fingerprint:
            recent_final_model = lgb.Booster(model_str=final_model_path.read_text(encoding="utf-8"))
            final_model_reused = True
            print("复用最终近期专家模型。")

    if not final_model_reused:
        print(f"训练最终近期专家 [{final_recent_start}, {VALID_STOP})。", flush=True)
        recent_final_model, final_train_info = train_ranker(
            [
                ("train", final_recent_start, TRAIN_STOP),
                ("valid", VALID_START, VALID_STOP),
            ],
            selected_recent_rounds,
        )
        save_model_text_atomic(recent_final_model, final_model_path)
        atomic_write_json(final_model_metadata_path, {
            "fingerprint": final_model_fingerprint,
            "train_start": final_recent_start,
            "train_stop": VALID_STOP,
            "rounds": selected_recent_rounds,
            **final_train_info,
        })

    raw_test_cache_path = RUNTIME_CACHE_DIR / f"raw_test_{final_model_fingerprint[:16]}.npy"
    if raw_test_cache_path.exists():
        raw_test_prediction = np.load(raw_test_cache_path)
        assert raw_test_prediction.shape == (int(common["test"]["groups"].sum()),)
        print("复用近期专家 Test 原始预测。")
    else:
        raw_test_prediction, test_groups = predict_interval(
            recent_final_model,
            "test",
            TEST_START,
            TEST_STOP,
            selected_recent_rounds,
        )
        atomic_save_npy(raw_test_cache_path, raw_test_prediction)

    test_groups = np.asarray(common["test"]["groups"], dtype=np.int32)
    test_stocks = np.asarray(common["test"]["stock"], dtype=np.int32)
    expert_grid = np.full((TEST_TIME_POINTS, STOCK_COUNT), 0.5, dtype=np.float32)
    prediction_grid = np.full((TEST_TIME_POINTS, STOCK_COUNT), 0.5, dtype=np.float32)

    offset = 0
    for local_time, size in enumerate(test_groups):
        size = int(size)
        stocks = test_stocks[offset:offset + size]
        expert_rank = rankdata(
            raw_test_prediction[offset:offset + size], method="average"
        ).astype(np.float32) / float(size)
        anchor_rank = rankdata(
            anchor_grid[local_time, stocks], method="average"
        ).astype(np.float32) / float(size)
        combined = (
            (1.0 - selected_recent_weight) * anchor_rank
            + selected_recent_weight * expert_rank
        )
        blended_rank = rankdata(combined, method="average").astype(np.float32) / float(size)
        expert_grid[local_time, stocks] = expert_rank
        prediction_grid[local_time, stocks] = blended_rank
        offset += size
    assert offset == raw_test_prediction.size == int(test_groups.sum())

    if selected_recent_weight == 0.0:
        prediction_grid = anchor_grid.copy()

    test_anchor_correlation = mean_cross_sectional_rank_correlation(
        prediction_grid,
        anchor_grid,
        test_groups,
        test_stocks,
    )

test_mask = np.zeros((TEST_TIME_POINTS, STOCK_COUNT), dtype=bool)
test_mask[
    np.asarray(common["test"]["time"], dtype=np.int32) - TEST_START,
    np.asarray(common["test"]["stock"], dtype=np.int32),
] = True

assert int(test_mask.sum()) == 2_042_538
assert prediction_grid.shape == expert_grid.shape == (TEST_TIME_POINTS, STOCK_COUNT)
assert prediction_grid.dtype == expert_grid.dtype == np.float32
assert np.isfinite(prediction_grid).all() and np.isfinite(expert_grid).all()
assert np.all(prediction_grid[~test_mask] == 0.5)
assert np.all(expert_grid[~test_mask] == 0.5)
assert float(prediction_grid.min()) >= 0.0 and float(prediction_grid.max()) <= 1.0

atomic_save_npy(RUN_DIR / "expert_prediction.npy", expert_grid)
atomic_save_npy(RUN_DIR / "prediction.npy", prediction_grid)

print(pd.Series({
    "prediction_shape": prediction_grid.shape,
    "prediction_dtype": str(prediction_grid.dtype),
    "evaluation_count": int(test_mask.sum()),
    "non_evaluation_count": int((~test_mask).sum()),
    "non_evaluation_all_0_5": bool(np.all(prediction_grid[~test_mask] == 0.5)),
    "minimum": float(prediction_grid.min()),
    "maximum": float(prediction_grid.max()),
    "mean": float(prediction_grid.mean()),
    "test_anchor_rank_correlation": test_anchor_correlation,
}))


## Checks

### 9. 保存统一结果并复读验收


In [ ]:
prediction_path = RUN_DIR / "prediction.npy"
expert_prediction_path = RUN_DIR / "expert_prediction.npy"
valid_prediction_path = RUN_DIR / "valid_prediction.npy"

loaded_prediction = np.load(prediction_path, mmap_mode="r")
assert loaded_prediction.shape == (TEST_TIME_POINTS, STOCK_COUNT)
assert loaded_prediction.dtype == np.float32
assert np.isfinite(loaded_prediction).all()
assert np.all(loaded_prediction[~test_mask] == 0.5)

test_correlation_gate_passed = bool(test_anchor_correlation >= MIN_TEST_ANCHOR_CORRELATION)
if RUN_MODE == "full" and not test_correlation_gate_passed:
    promoted = False

existing_metrics_path = RUN_DIR / "metrics.json"
existing_metadata_path = RUN_DIR / "metadata.json"
existing_metrics = (
    json.loads(existing_metrics_path.read_text(encoding="utf-8"))
    if existing_metrics_path.exists()
    else {}
)
existing_metadata = (
    json.loads(existing_metadata_path.read_text(encoding="utf-8"))
    if existing_metadata_path.exists()
    else {}
)
recorded_online_rank_ic = existing_metrics.get(
    "online_rank_ic",
    existing_metadata.get("online_rank_ic"),
)

status = (
    "preflight_only"
    if RUN_MODE == "preflight"
    else (
        "submitted_online_best"
        if recorded_online_rank_ic is not None
        else ("completed_candidate" if promoted else "completed_not_promoted")
    )
)

selected_internal_rows = weight_search[weight_search["selected"]]
selected_internal_metrics = (
    selected_internal_rows.iloc[0].to_dict()
    if not selected_internal_rows.empty
    else {}
)

metrics = {
    "run_mode": RUN_MODE,
    "status": status,
    "selected_recent_rounds": selected_recent_rounds,
    "selected_recent_weight": selected_recent_weight,
    "internal_selection": selected_internal_metrics,
    "official_valid_anchor": official_base_metrics,
    "official_valid_expert": official_expert_metrics,
    "official_valid_blend": official_blend_metrics,
    "official_mean_delta": official_mean_delta,
    "official_late_delta": official_late_delta,
    "official_worst_quarter_delta": official_worst_delta,
    "anchor_reproduced": official_anchor_reproduced,
    "test_anchor_rank_correlation": test_anchor_correlation,
    "test_correlation_gate_passed": test_correlation_gate_passed,
    "promoted": promoted,
}
if recorded_online_rank_ic is not None:
    metrics.update({
        "online_rank_ic": float(recorded_online_rank_ic),
        "online_result_source": existing_metrics.get(
            "online_result_source",
            "preserved_existing_record",
        ),
    })

prediction_sha256 = file_sha256(prediction_path)
expert_sha256 = file_sha256(expert_prediction_path)
metadata = {
    "experiment_id": EXPERIMENT_ID,
    "run_mode": RUN_MODE,
    "status": status,
    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_fingerprint": RUN_FINGERPRINT,
    "manifest_sha256": manifest_sha256,
    "anchor_path": str(ANCHOR_PATH),
    "anchor_sha256": file_sha256(ANCHOR_PATH),
    "prediction_path": str(prediction_path),
    "prediction_sha256": prediction_sha256,
    "expert_prediction_sha256": expert_sha256,
    "shape": [TEST_TIME_POINTS, STOCK_COUNT],
    "dtype": "float32",
    "finite": True,
    "evaluation_count": int(test_mask.sum()),
    "non_evaluation_count": int((~test_mask).sum()),
    "non_evaluation_value": 0.5,
    "minimum": float(loaded_prediction.min()),
    "maximum": float(loaded_prediction.max()),
    "mean": float(loaded_prediction.mean()),
    "selected_recent_rounds": selected_recent_rounds,
    "selected_recent_weight": selected_recent_weight,
    "selection_reason": selection_reason,
    "promoted": promoted,
    "formal_submission_overwritten": False,
}
if recorded_online_rank_ic is not None:
    metadata.update({
        "online_rank_ic": float(recorded_online_rank_ic),
        "online_status": existing_metadata.get("online_status", "submitted"),
        "online_recorded_at": existing_metadata.get("online_recorded_at"),
    })

report_lines = [
    "# 锚点与近期专家走步融合实验报告",
    "",
    f"- 运行模式：`{RUN_MODE}`",
    f"- 状态：`{status}`",
    f"- 特征视图：`legacy_328`",
    f"- 锚点：`{ANCHOR_PATH}`",
    f"- 近期窗口：`{RECENT_LOOKBACK}` 期",
    f"- 选中近期轮数：`{selected_recent_rounds}`",
    f"- 选中近期权重：`{selected_recent_weight:.2f}`",
    f"- 官方 Valid 锚点 RankIC：`{official_base_metrics['mean_rank_ic']:.6f}`",
    f"- 官方 Valid 融合 RankIC：`{official_blend_metrics['mean_rank_ic']:.6f}`",
    f"- 官方 Valid 增量：`{official_mean_delta:+.6f}`",
    f"- Test 与正式锚点平均截面秩相关：`{test_anchor_correlation:.6f}`",
    f"- 是否晋级：`{promoted}`",
    f"- 预测 SHA-256：`{prediction_sha256}`",
    "",
    "## 选择说明",
    "",
    selection_reason,
    "",
    "## 产物",
    "",
    "- `prediction.npy`：本实验完整 Test 结果。",
    "- `expert_prediction.npy`：近期专家逐时间排名结果。",
    "- `valid_prediction.npy`：官方 Valid 选中融合结果。",
    "- `fold_results.csv`：各折、轮数与权重的明细。",
    "- `weight_search.csv`：稳定性门槛和最终选择。",
    "- `official_valid_results.csv`：官方 Valid 一次性检查。",
    "",
    "> 本实验不会自动覆盖 `04_results/final_submission/prediction.npy`。",
]
if recorded_online_rank_ic is not None:
    report_lines.extend([
        "",
        "## 已记录线上结果",
        "",
        f"- 线上 RankIC：`{float(recorded_online_rank_ic):.6f}`",
        "- 重新运行 Notebook 时会保留该外部记录。",
    ])

atomic_write_json(RUN_DIR / "metrics.json", metrics)
atomic_write_json(RUN_DIR / "metadata.json", metadata)
atomic_write_text(RUN_DIR / "experiment_report.md", "\n".join(report_lines) + "\n")

required_outputs = [
    RUN_DIR / "prediction.npy",
    RUN_DIR / "expert_prediction.npy",
    RUN_DIR / "valid_prediction.npy",
    RUN_DIR / "fold_results.csv",
    RUN_DIR / "weight_search.csv",
    RUN_DIR / "official_valid_results.csv",
    RUN_DIR / "metrics.json",
    RUN_DIR / "metadata.json",
    RUN_DIR / "experiment_report.md",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"结果文件不完整：{missing_outputs}")

print("实验运行完成，结果目录：", RUN_DIR)
print("prediction.npy：", prediction_path)
print("状态：", status)


## Takeaways

运行完成后，以底部代码单元输出和 `experiment_report.md` 为准：

- `completed_candidate`：候选通过内部稳定性、官方 Valid 和 Test 锚点相关性门槛，可进入人工提交评估。
- `completed_not_promoted`：结果文件完整生成，但不建议替换正式提交。
- `preflight_only`：仅证明数据契约、选择逻辑和文件写出可用，不代表模型成绩。
